In [1]:
import os
import requests
from dotenv import load_dotenv
from neo4j import GraphDatabase, basic_auth
from typing import List
from neo4j_graphrag.embeddings.base import Embedder
from neo4j_graphrag.retrievers import VectorRetriever


load_dotenv()

driver = GraphDatabase.driver(
  os.getenv('neo4j_url'),
  auth=basic_auth("neo4j", os.getenv('neo4j_name')))


class HyperClovaXEmbeddings(Embedder):
    def __init__(self):
        self.url = "https://clovastudio.stream.ntruss.com/testapp/v1/api-tools/embedding/v2"
        api_key = os.getenv('CLOVA_API_KEY')
        if not api_key:
            raise ValueError("CLOVA_API_KEY 환경 변수가 설정되지 않았습니다.")
        
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        }

    def embed_query(self, text: str) -> List[float]:
        data = {"text": text}
        try:
            response = requests.post(self.url, headers=self.headers, json=data)
            response.raise_for_status() 
            result = response.json()
            
            if result and "result" in result and "embedding" in result["result"]:
                return result["result"]["embedding"]
            else:
                print(f"경고: API 응답에서 임베딩을 찾을 수 없습니다. 응답: {result}")
                return []
        except requests.exceptions.RequestException as e:
            print(f"API 요청 중 오류 발생: {e}")
            return []

embedder = HyperClovaXEmbeddings()

retriever = VectorRetriever(
    driver,
    index_name='moviePlotsEmbedding',
    embedder=embedder,
    return_properties=['title', 'plot']
)


In [47]:
import os
import requests
from typing import List, Optional, Dict, Any
from neo4j_graphrag.generation import GraphRAG
from types import SimpleNamespace

class HyperClovaXLLM:
    def __init__(
        self,
        model_name: str = "HCX-003",
        api_key: Optional[str] = None,
        base_url: str = "https://clovastudio.stream.ntruss.com/testapp/v1",
        model_params: Optional[Dict[str, Any]] = None,
    ):
        self.model_name = model_name
        self.api_key = api_key or os.getenv("CLOVA_API_KEY")
        if not self.api_key:
            raise ValueError("CLOVA_API_KEY 환경변수가 필요합니다.")
        self.base_url = base_url
        self.model_params = model_params or {}

    @property
    def headers(self):
        return {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

    def get_messages(
        self,
        input: str,
        message_history: Optional[List[Dict[str, str]]] = None,
        system_instruction: Optional[str] = None,
    ) -> List[Dict[str, str]]:
        messages = []
        if system_instruction:
            messages.append({"role": "system", "content": system_instruction})
        if message_history:
            messages.extend(message_history)
        messages.append({"role": "user", "content": input})
        return messages

    def invoke(
        self,
        input: str,
        message_history: Optional[List[Dict[str, str]]] = None,
        system_instruction: Optional[str] = None,
    ) -> str:
        url = f"{self.base_url}/chat-completions/{self.model_name}"
        data = {
            "messages": self.get_messages(input, message_history, system_instruction),
            "maxTokens": self.model_params.get("maxTokens", 1000),
            "temperature": self.model_params.get("temperature", 0.7),
        }
        response = requests.post(url, headers=self.headers, json=data)
        response.raise_for_status()
        result = response.json()
        # 응답 구조에 따라 content 추출

        if "result" in result and "message" in result["result"]:
            return SimpleNamespace(**result["result"]["message"])
        elif "choices" in result and len(result["choices"]) > 0:
            return SimpleNamespace(**result["choices"][0]["message"])
        else:
            return str(result)


In [48]:
llm = HyperClovaXLLM(model_name='HCX-003')

In [49]:
rag = GraphRAG(retriever=retriever, llm=llm)

In [50]:
query_text = 'What movies are sad romances?'
response = rag.search(query_text=query_text, retriever_config={'top_k': 5})
print(response.answer)

c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\neo4j_graphrag\generation\graphrag.py:120: DeprecationWarning: The default value of 'return_context' will change from 'False' to 'True' in a future version.
  warnings.warn(
c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\neo4j_graphrag\retrievers\vector.py:201: DeprecationWarning: The default returned 'id' field in the search results will be removed. Please switch to using 'elementId' instead.
  search_query, search_params = get_search_query(


Based on the provided context, the movie "Bed of Roses" can be considered a sad romance as it is a romantic drama about a young career girl who falls in love with a shy florist, but there isn't enough information to determine if the ending is happy or sad.


In [ ]:
b